In [2]:
import pandas as pd
import matplotlib.pyplot as plt


In [3]:
df = pd.read_csv("ec2_cpu_utilization_5f5533.csv")

In [ ]:
print(f"shape: {df.shape}")
print(df.head())
print("*"*25 + "Missing Values" + "*"*25)
print(df[df.isna()].sum()) # no missing values

shape: (4032, 2)
             timestamp   value
0  2014-02-14 14:27:00  51.846
1  2014-02-14 14:32:00  44.508
2  2014-02-14 14:37:00  41.244
3  2014-02-14 14:42:00  48.568
4  2014-02-14 14:47:00  46.714
*************************Missing Values*************************
timestamp      0
value        0.0
dtype: object


In [11]:
print(df.info())
print("="*50)
print(str(df["value"].nunique()) + " -- many repeated values")
print("="*50)
print(df["timestamp"].nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4032 entries, 0 to 4031
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   timestamp  4032 non-null   object 
 1   value      4032 non-null   float64
dtypes: float64(1), object(1)
memory usage: 63.1+ KB
None
2118 -- many repeated values
4032


In [30]:
df.describe()
mean = df["value"].mean()
sd = df["value"].std()
min = df["value"].min()
max = df["value"].max()

Mean: 43.110
Median: 42.918
STD: 4.304

**Rolling Average**

In [46]:
# 144 window, 5 mins, rolling avg every 12 hrs
df["12_hr_avg"] = df["value"].rolling(window=144).mean()
print(df["12_hr_avg"])
df["12_hr_sd"] = df["value"].rolling(window=144).std()
print(df["12_hr_sd"])

0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
          ...    
4027    38.315375
4028    38.326708
4029    38.324431
4030    38.334097
4031    38.334708
Name: 12_hr_avg, Length: 4032, dtype: float64
0            NaN
1            NaN
2            NaN
3            NaN
4            NaN
          ...   
4027    0.959422
4028    0.973767
4029    0.974355
4030    0.968671
4031    0.968251
Name: 12_hr_sd, Length: 4032, dtype: float64


**Eyeballing Anomalies**

In [26]:
Q1 = df["value"].quantile(0.25)
Q3 = df["value"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - IQR*1.5
upper_bound = Q3 + IQR*1.5
print(f"=====BOUNDARIES=====")
print(f"Q1: {Q1}")
print(f"Q3: {Q3}")
print(f"=====OUTLIERS=====")
print(f"Below Q1: {len(df[df['value'] < lower_bound])}")
print(f"Above Q3: {len(df[df['value'] > upper_bound])}")
print("=====Lowest=====")
print(f"Lowest: {df["value"].min()}")
print("=====Particularly high values=====")
print(df[df['value'] > upper_bound])

=====BOUNDARIES=====
Q1: 39.296
Q3: 46.006
=====OUTLIERS=====
Below Q1: 0
Above Q3: 4
=====Lowest=====
Lowest: 34.766
=====Particularly high values=====
                timestamp   value
444   2014-02-16 03:27:00  56.220
918   2014-02-17 18:57:00  56.408
1271  2014-02-19 00:22:00  62.056
2970  2014-02-24 21:57:00  68.092


**Using Standard Deviation**

In [43]:
print(f"*****Outside 1 SD of the mean***** [{mean-1*sd:.3f}, {mean+1*sd:.3f}]")
print(df[(df["value"] < mean-1*sd) | (df["value"] > mean+1*sd)])
print(f"*****Outside 2 SD of the mean***** [{mean-2*sd:.3f}, {mean+1*sd:.2f}]")
print(df[(df["value"] < mean-2*sd) | (df["value"] > mean+2*sd)])
print(f"*****Outside 3 SD of the mean***** [{mean-3*sd:.3f}, {mean+3*sd:.3f}]")
print(df[(df["value"] < mean-3*sd) | (df["value"] > mean+3*sd)])

*****Outside 1 SD of the mean***** [38.807, 47.414]
                timestamp   value
0     2014-02-14 14:27:00  51.846
3     2014-02-14 14:42:00  48.568
6     2014-02-14 14:57:00  49.108
8     2014-02-14 15:07:00  53.404
11    2014-02-14 15:22:00  49.720
...                   ...     ...
4025  2014-02-28 13:52:00  37.090
4027  2014-02-28 14:02:00  38.474
4029  2014-02-28 14:12:00  37.912
4030  2014-02-28 14:17:00  38.458
4031  2014-02-28 14:22:00  37.718

[1544 rows x 2 columns]
*****Outside 2 SD of the mean***** [34.503, 47.41]
                timestamp   value
0     2014-02-14 14:27:00  51.846
8     2014-02-14 15:07:00  53.404
24    2014-02-14 16:27:00  52.586
35    2014-02-14 17:22:00  52.606
43    2014-02-14 18:02:00  52.940
...                   ...     ...
1278  2014-02-19 00:57:00  54.058
1291  2014-02-19 02:02:00  52.422
1328  2014-02-19 05:07:00  52.184
1878  2014-02-21 02:57:00  51.830
2970  2014-02-24 21:57:00  68.092

[151 rows x 2 columns]
*****Outside 3 SD of the mean***